지금 캔버스 코드가 하는 일 (네 체크리스트 그대로)

1. cam0 / cam1 ROI 좌표 고정 ✔
2. 이미지 로드 → ROI 자르기 ✔
3. ROI 안에서 보드 후보 컨투어 탐색 ✔
4. 54×39 비율(1.3846) 가장 가까운 사각형을 “정답”으로 선택 ✔
5. 그 사각형의 픽셀 가로·세로 → px/cm 계산 ✔
6. 하위 폴더까지 전부 탐색 ✔
7. ThreadPoolExecutor 병렬 처리 ✔
  tqdm으로 남은 시간/진행률 표시 ✔
8. 결과를 **CSV(px/cm 형태)**로 저장 ✔



In [5]:
# ==========================================================
# ROI 기반 보드 검출 → pixel_scale(px/cm) 계산 파이프라인
# cam0 / cam1 공통 로직 + cam별 ROI만 분기
# 하위 폴더 전체 탐색, 병렬 처리, 진행률 표시(tqdm)
# ==========================================================

import os
import cv2
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# =========================
# 1. 설정값
# =========================

# 실제 보드 크기 (cm)
BOARD_W_CM = 54.0  # 가로
BOARD_H_CM = 39.0  # 세로
TARGET_RATIO = BOARD_W_CM / BOARD_H_CM

# cam별 ROI (정규화 좌표)
ROI_CONFIG = {
    0: {"x": (0.50, 0.95), "y": (0.55, 0.95)},  # cam0
    1: {"x": (0.05, 0.50), "y": (0.55, 0.95)},  # cam1
}

# 병렬 워커 수 (CPU 상황에 맞게 조절)
N_WORKERS = max(4, os.cpu_count() // 2)

# =========================
# 2. 유틸 함수
# =========================

def get_cam_id(filename: str) -> int:
    """
    파일명에서 cam id 추출 (cam0 / cam1)
    예: bed00_20251213_062821_cam1.jpg
    """
    if "cam1" in filename:
        return 1
    return 0


def crop_roi(img, cam_id):
    h, w = img.shape[:2]
    cfg = ROI_CONFIG[cam_id]
    x1, x2 = int(cfg["x"][0] * w), int(cfg["x"][1] * w)
    y1, y2 = int(cfg["y"][0] * h), int(cfg["y"][1] * h)
    return img[y1:y2, x1:x2], (x1, y1)


# =========================
# 3. 보드 검출 로직 (공통)
# =========================

DEBUG_SAVE_FAILED = False          # 실패 케이스 ROI/마스크 저장할지
DEBUG_DIR = "_debug_board_fail"   # 저장 폴더
MAX_DEBUG_SAVE = 80               # 너무 많이 저장되지 않게 제한
_debug_saved = 0


def _ensure_dir(p):
    os.makedirs(p, exist_ok=True)


def detect_board_in_roi(roi_bgr):
    """
    ROI 이미지에서 '검은 보드' 후보를 찾고
    best 후보의 (long_px, short_px, px_per_cm_x, px_per_cm_y, best_cnt, mask) 반환
    실패 시 None 반환

    핵심 포인트
    - Otsu(밝기 기반)만 쓰면 잎/바닥 텍스처 때문에 실패가 많음
    - HSV에서 '어두움 + 저채도' 조건으로 블랙보드만 강하게 마스킹
    """
    h, w = roi_bgr.shape[:2]
    roi_area = float(h * w)

    hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
    H, S, V = cv2.split(hsv)

    # 블랙보드 후보 마스크: 어둡고(V 낮음) + 채도 낮음(S 낮음)
    # (임계는 현장 노출에 따라 약간 조정 가능)
    mask_dark = (V < 85).astype(np.uint8) * 255
    mask_low_sat = (S < 90).astype(np.uint8) * 255
    mask = cv2.bitwise_and(mask_dark, mask_low_sat)

    # 보드가 약간 밝게 뜨는 경우 대비: 너무 빡빡하면 놓침 → 완화용
    # mask2 = (V < 105) & (S < 120) 도 가능하지만, 우선 위 기준으로 시작

    kernel = np.ones((9, 9), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best = None
    best_score = -1e18

    # 면적 하한: ROI의 0.5% 미만은 버림 (환경 잡영 제거)
    min_area = 0.005 * roi_area

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue

        rect = cv2.minAreaRect(cnt)
        (rw, rh) = rect[1]
        if rw <= 1 or rh <= 1:
            continue

        long_px = float(max(rw, rh))
        short_px = float(min(rw, rh))
        ratio = long_px / short_px

        # 비율 근접도(가장 중요): 목표 1.3846
        ratio_err = abs(ratio - TARGET_RATIO)
        ratio_score = -ratio_err  # 작을수록 좋음

        # 직사각형 충실도(너무 빡세면 실패 급증하므로 완화)
        rect_area = float(rw * rh)
        fill_ratio = area / rect_area if rect_area > 0 else 0.0
        if fill_ratio < 0.55:
            continue

        # 너무 길쭉/납작한 잡영 제거 (보드 비율 근처만)
        if not (1.10 <= ratio <= 1.75):
            continue

        # 점수: 비율 + 충실도 + 면적(조금 가중)
        # 면적은 log로 완만하게 반영
        score = (ratio_score * 3.0) + (fill_ratio * 1.5) + (np.log(area + 1.0) * 0.25)

        if score > best_score:
            best_score = score
            px_per_cm_x = long_px / BOARD_W_CM
            px_per_cm_y = short_px / BOARD_H_CM
            best = (long_px, short_px, px_per_cm_x, px_per_cm_y, cnt)

    if best is None:
        return None, mask

    return best, mask


# =========================
# 4. 단일 이미지 처리
# =========================

def process_image(img_path):
    global _debug_saved

    try:
        img = cv2.imread(img_path)
        if img is None:
            return None

        cam_id = get_cam_id(os.path.basename(img_path))
        roi_img, (x_off, y_off) = crop_roi(img, cam_id)

        best, mask = detect_board_in_roi(roi_img)
        if best is None:
            # 디버그 저장(필요할 때만)
            if DEBUG_SAVE_FAILED and _debug_saved < MAX_DEBUG_SAVE:
                _ensure_dir(DEBUG_DIR)
                base = os.path.splitext(os.path.basename(img_path))[0]
                cv2.imwrite(os.path.join(DEBUG_DIR, f"{base}_roi.jpg"), roi_img)
                cv2.imwrite(os.path.join(DEBUG_DIR, f"{base}_mask.png"), mask)
                _debug_saved += 1
            return None

        long_px, short_px, px_per_cm_x, px_per_cm_y, _ = best

        return {
            "image_name": os.path.basename(img_path),
            "cam": cam_id,
            "width_px": long_px,
            "height_px": short_px,
            "px_per_cm_x": px_per_cm_x,
            "px_per_cm_y": px_per_cm_y,
        }

    except Exception:
        return None


# =========================
# 5. 전체 실행
# =========================

def run_pixel_scale_extraction(root_dir, output_csv):
    # 이미지 파일 전체 수집
    img_files = []
    for root, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith((".jpg", ".png")):
                img_files.append(os.path.join(root, f))

    results = []

    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = [executor.submit(process_image, p) for p in img_files]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing"):
            res = future.result()
            if res is not None:
                results.append(res)

    # CSV 저장
    import pandas as pd
    df = pd.DataFrame(results,
                      columns=["image_name","cam","width_px","height_px","px_per_cm_x","px_per_cm_y"])
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print(f"Done: {len(df)} / {len(img_files)} images succeeded")



In [6]:

# =========================
# 6. 실행 예시
# =========================
if __name__ == "__main__":
    # 날짜 1개 폴더만 테스트
    TEST_ROOT = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251213-251226/251213"   # ← 네 경로로 수정
    OUTPUT_CSV = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251213-251226/260102_pixel_scale_20251213.csv"

    run_pixel_scale_extraction(TEST_ROOT, OUTPUT_CSV)


Processing: 100%|██████████| 1676/1676 [01:36<00:00, 17.41it/s]

Done: 1070 / 1676 images succeeded


In [9]:
import pandas as pd

df = pd.read_csv("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251213-251226/260102_pixel_scale_20251213.csv")

def cv(x):
    return (x.std() / x.mean()) * 100

# mean, std, cv 계산
df.groupby("cam")[["px_per_cm_x","px_per_cm_y"]].agg(["mean","std", cv])

px_per_cm_x                      px_per_cm_y                     
           mean       std         cv        mean       std         cv
cam                                                                  
0      8.993750  0.774522   8.611782    9.846646  0.460489   4.676604
1      9.186473  3.158157  34.378342    9.297101  3.148879  33.869471

### CAM1 코드 투시보정이 되도록 한번 더

In [10]:
# ==========================================================
# ROI 기반 보드 검출 → pixel_scale(px/cm) 계산 파이프라인
# cam0 / cam1 공통 로직 + cam별 ROI만 분기
# 하위 폴더 전체 탐색, 병렬 처리, 진행률 표시(tqdm)
# ==========================================================

import os
import cv2
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# =========================
# 1. 설정값
# =========================

# 실제 보드 크기 (cm)
BOARD_W_CM = 54.0  # 가로
BOARD_H_CM = 39.0  # 세로
TARGET_RATIO = BOARD_W_CM / BOARD_H_CM

# cam별 ROI (정규화 좌표)
ROI_CONFIG = {
    0: {"x": (0.50, 0.95), "y": (0.55, 0.95)},  # cam0
    1: {"x": (0.05, 0.50), "y": (0.55, 0.95)},  # cam1
}

# 병렬 워커 수 (CPU 상황에 맞게 조절)
N_WORKERS = max(4, os.cpu_count() // 2)

# =========================
# 2. 유틸 함수
# =========================

def get_cam_id(filename: str) -> int:
    """
    파일명에서 cam id 추출 (cam0 / cam1)
    예: bed00_20251213_062821_cam1.jpg
    """
    if "cam1" in filename:
        return 1
    return 0


def crop_roi(img, cam_id):
    h, w = img.shape[:2]
    cfg = ROI_CONFIG[cam_id]
    x1, x2 = int(cfg["x"][0] * w), int(cfg["x"][1] * w)
    y1, y2 = int(cfg["y"][0] * h), int(cfg["y"][1] * h)
    return img[y1:y2, x1:x2], (x1, y1)


# =========================
# 3. 보드 검출 로직 (공통)
# =========================

DEBUG_SAVE_FAILED = False          # 실패 케이스 ROI/마스크 저장할지
DEBUG_DIR = "_debug_board_fail"   # 저장 폴더
MAX_DEBUG_SAVE = 80               # 너무 많이 저장되지 않게 제한
_debug_saved = 0


def _ensure_dir(p):
    os.makedirs(p, exist_ok=True)


def order_points_clockwise(pts: np.ndarray) -> np.ndarray:
    """pts: (4,2) -> (4,2) ordered as TL, TR, BR, BL"""
    pts = np.asarray(pts, dtype=np.float32)
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).reshape(-1)
    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(diff)]
    bl = pts[np.argmax(diff)]
    return np.stack([tl, tr, br, bl], axis=0)


def corners_from_contour(cnt: np.ndarray) -> np.ndarray:
    """컨투어에서 4코너 추출. 실패 시 minAreaRect 박스포인트로 대체."""
    peri = cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
    if len(approx) == 4:
        pts = approx.reshape(4, 2)
        return order_points_clockwise(pts)

    # fallback
    rect = cv2.minAreaRect(cnt)
    box = cv2.boxPoints(rect)  # 4x2
    return order_points_clockwise(box)


def scale_from_hinv(Hinv: np.ndarray, center_cm=(BOARD_W_CM/2, BOARD_H_CM/2)):
    """cm->pixel 변환(Hinv)으로 중심점에서 px/cm를 근사 계산"""
    cx, cy = center_cm
    p0 = np.array([[[cx, cy]]], dtype=np.float32)
    px = np.array([[[cx + 1.0, cy]]], dtype=np.float32)  # +1 cm in x
    py = np.array([[[cx, cy + 1.0]]], dtype=np.float32)  # +1 cm in y

    ip0 = cv2.perspectiveTransform(p0, Hinv)[0, 0]
    ipx = cv2.perspectiveTransform(px, Hinv)[0, 0]
    ipy = cv2.perspectiveTransform(py, Hinv)[0, 0]

    px_per_cm_x = float(np.linalg.norm(ipx - ip0))
    px_per_cm_y = float(np.linalg.norm(ipy - ip0))
    return px_per_cm_x, px_per_cm_y


def detect_board_in_roi(roi_bgr):
    """
    ROI 이미지에서 '검은 보드' 후보를 찾고 best 후보 반환

    반환:
      best_dict: {
        long_px, short_px, px_per_cm_x, px_per_cm_y,
        corners (TL,TR,BR,BL in ROI coords),
        H_pix2cm (pixel->cm), H_cm2pix
      }
      mask

    검출은 HSV(어두움+저채도) 기반.
    """
    h, w = roi_bgr.shape[:2]
    roi_area = float(h * w)

    hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
    Hc, S, V = cv2.split(hsv)

    # 블랙보드 후보 마스크: 어둡고(V 낮음) + 채도 낮음(S 낮음)
    mask_dark = (V < 85).astype(np.uint8) * 255
    mask_low_sat = (S < 90).astype(np.uint8) * 255
    mask = cv2.bitwise_and(mask_dark, mask_low_sat)

    kernel = np.ones((9, 9), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best = None
    best_score = -1e18

    # 면적 하한: ROI의 0.5% 미만은 버림
    min_area = 0.005 * roi_area

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue

        rect = cv2.minAreaRect(cnt)
        (rw, rh) = rect[1]
        if rw <= 1 or rh <= 1:
            continue

        long_px = float(max(rw, rh))
        short_px = float(min(rw, rh))
        ratio = long_px / short_px

        ratio_err = abs(ratio - TARGET_RATIO)
        rect_area = float(rw * rh)
        fill_ratio = area / rect_area if rect_area > 0 else 0.0
        if fill_ratio < 0.55:
            continue

        if not (1.10 <= ratio <= 1.75):
            continue

        score = (-ratio_err * 3.0) + (fill_ratio * 1.5) + (np.log(area + 1.0) * 0.25)

        if score > best_score:
            best_score = score

            # 코너 4점 추출 (ROI 좌표)
            corners = corners_from_contour(cnt)  # TL,TR,BR,BL

            # homography: pixel(ROI) -> cm
            dst_cm = np.array([
                [0.0, 0.0],
                [BOARD_W_CM, 0.0],
                [BOARD_W_CM, BOARD_H_CM],
                [0.0, BOARD_H_CM]
            ], dtype=np.float32)

            H_pix2cm = cv2.getPerspectiveTransform(corners.astype(np.float32), dst_cm)
            H_cm2pix = np.linalg.inv(H_pix2cm)

            # 기본(minAreaRect) 기반 px/cm (fallback)
            px_per_cm_x_rect = long_px / BOARD_W_CM
            px_per_cm_y_rect = short_px / BOARD_H_CM

            best = {
                "long_px": long_px,
                "short_px": short_px,
                "px_per_cm_x_rect": px_per_cm_x_rect,
                "px_per_cm_y_rect": px_per_cm_y_rect,
                "corners": corners,
                "H_pix2cm": H_pix2cm,
                "H_cm2pix": H_cm2pix,
            }

    if best is None:
        return None, mask

    # homography 기반 중심점 px/cm 계산
    try:
        px_per_cm_x_h, px_per_cm_y_h = scale_from_hinv(best["H_cm2pix"])
        best["px_per_cm_x_h"] = px_per_cm_x_h
        best["px_per_cm_y_h"] = px_per_cm_y_h
    except Exception:
        best["px_per_cm_x_h"] = None
        best["px_per_cm_y_h"] = None

    return best, mask


# =========================
# 4. 단일 이미지 처리
# =========================

def process_image(img_path):
    global _debug_saved

    try:
        img = cv2.imread(img_path)
        if img is None:
            return None

        cam_id = get_cam_id(os.path.basename(img_path))
        roi_img, (x_off, y_off) = crop_roi(img, cam_id)

        best, mask = detect_board_in_roi(roi_img)
        if best is None:
            # 디버그 저장(필요할 때만)
            if DEBUG_SAVE_FAILED and _debug_saved < MAX_DEBUG_SAVE:
                _ensure_dir(DEBUG_DIR)
                base = os.path.splitext(os.path.basename(img_path))[0]
                cv2.imwrite(os.path.join(DEBUG_DIR, f"{base}_roi.jpg"), roi_img)
                cv2.imwrite(os.path.join(DEBUG_DIR, f"{base}_mask.png"), mask)
                _debug_saved += 1
            return None

        # cam1은 homography 기반 px/cm 우선 사용
        if cam_id == 1 and best.get("px_per_cm_x_h") is not None:
            px_per_cm_x = best["px_per_cm_x_h"]
            px_per_cm_y = best["px_per_cm_y_h"]
        else:
            px_per_cm_x = best["px_per_cm_x_rect"]
            px_per_cm_y = best["px_per_cm_y_rect"]

        long_px = best["long_px"]
        short_px = best["short_px"]

        return {
            "image_name": os.path.basename(img_path),
            "cam": cam_id,
            "width_px": long_px,
            "height_px": short_px,
            "px_per_cm_x": px_per_cm_x,
            "px_per_cm_y": px_per_cm_y,
        }

    except Exception:
        return None


# =========================
# 5. 전체 실행
# =========================

def run_pixel_scale_extraction(root_dir, output_csv):
    # 이미지 파일 전체 수집
    img_files = []
    for root, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith((".jpg", ".png")):
                img_files.append(os.path.join(root, f))

    results = []

    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = [executor.submit(process_image, p) for p in img_files]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing"):
            res = future.result()
            if res is not None:
                results.append(res)

    # CSV 저장
    import pandas as pd
    df = pd.DataFrame(results,
                      columns=["image_name","cam","width_px","height_px","px_per_cm_x","px_per_cm_y"])
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print(f"Done: {len(df)} / {len(img_files)} images succeeded")


# =========================
# 6. 실행 예시
# =========================
if __name__ == "__main__":
    # 날짜 1개 폴더만 테스트
    TEST_ROOT = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251213-251226/251213"   # ← 네 경로로 수정
    OUTPUT_CSV = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251213-251226/251213_pixel_scale_Cam1homography.csv"

    run_pixel_scale_extraction(TEST_ROOT, OUTPUT_CSV)


Processing: 100%|██████████| 1676/1676 [01:41<00:00, 16.43it/s]

Done: 1070 / 1676 images succeeded


In [11]:
df1=pd.read_csv("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251213-251226/251213_pixel_scale_Cam1homography.csv")
def cv(x):
    return (x.std() / x.mean()) * 100

# mean, std, cv 계산
df1.groupby("cam")[["px_per_cm_x","px_per_cm_y"]].agg(["mean","std", cv])

px_per_cm_x                      px_per_cm_y                     
           mean       std         cv        mean       std         cv
cam                                                                  
0      9.003064  0.769572   8.547894    9.845512  0.460289   4.675120
1      9.076926  3.390038  37.347863    9.437903  2.870718  30.416901

In [14]:
import re

# cam=1인 데이터 필터링
df_cam1 = df1[df1['cam'] == 1].copy()

# image_name에서 'bedXX_YYYYMMDD' 추출하여 새로운 컬럼 생성
df_cam1['bed_date'] = df_cam1['image_name'].apply(lambda x: re.match(r'(bed\d{2}_\d{8})', x).group(1) if re.match(r'(bed\d{2}_\d{8})', x) else None)

# bed_date로 그룹화하고 px_per_cm_x, px_per_cm_y의 평균 계산 후 오름차순 정렬
result = df_cam1.groupby('bed_date')[['px_per_cm_x', 'px_per_cm_y']].mean().sort_values(by='px_per_cm_x', ascending=True)

result.round(2)

,px_per_cm_x,px_per_cm_y
bed_date,,
bed25_20251213,1.43,3.00
bed73_20251213,1.52,3.26
bed52_20251213,1.72,2.97
bed26_20251213,2.36,1.89
bed86_20251213,2.52,2.06
...,...,...
bed03_20251213,11.30,10.72
bed20_20251213,11.44,11.39
bed94_20251213,11.47,10.44


###3. 찐마지막 cam1은 베드-날짜 기준으로 수정 <<<< 이걸로 낙찰***

In [2]:
# ==========================================================
# ROI 기반 보드 검출 → pixel_scale(px/cm) 계산 파이프라인
# cam0 / cam1 공통 로직 + cam별 ROI만 분기
# 하위 폴더 전체 탐색, 병렬 처리, 진행률 표시(tqdm)
# ==========================================================

import os
import cv2
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# =========================
# 1. 설정값
# =========================

# 실제 보드 크기 (cm)
BOARD_W_CM = 54.0  # 가로
BOARD_H_CM = 39.0  # 세로
TARGET_RATIO = BOARD_W_CM / BOARD_H_CM

# cam별 ROI (정규화 좌표)
ROI_CONFIG = {
    0: {"x": (0.50, 0.95), "y": (0.55, 0.95)},  # cam0
    1: {"x": (0.05, 0.50), "y": (0.55, 0.95)},  # cam1
}

# 병렬 워커 수 (CPU 상황에 맞게 조절)
N_WORKERS = max(8, os.cpu_count() // 2)

# =========================
# 2. 유틸 함수
# =========================

def get_cam_id(filename: str) -> int:
    """
    파일명에서 cam id 추출 (cam0 / cam1)
    예: bed00_20251213_062821_cam1.jpg
    """
    if "cam1" in filename:
        return 1
    return 0


def crop_roi(img, cam_id):
    h, w = img.shape[:2]
    cfg = ROI_CONFIG[cam_id]
    x1, x2 = int(cfg["x"][0] * w), int(cfg["x"][1] * w)
    y1, y2 = int(cfg["y"][0] * h), int(cfg["y"][1] * h)
    return img[y1:y2, x1:x2], (x1, y1)


# =========================
# 3. 보드 검출 로직 (공통)
# =========================

DEBUG_SAVE_FAILED = False          # 실패 케이스 ROI/마스크 저장할지
DEBUG_DIR = "_debug_board_fail"   # 저장 폴더
MAX_DEBUG_SAVE = 80               # 너무 많이 저장되지 않게 제한
_debug_saved = 0


def _ensure_dir(p):
    os.makedirs(p, exist_ok=True)


def order_points_clockwise(pts: np.ndarray) -> np.ndarray:
    """pts: (4,2) -> (4,2) ordered as TL, TR, BR, BL"""
    pts = np.asarray(pts, dtype=np.float32)
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).reshape(-1)
    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(diff)]
    bl = pts[np.argmax(diff)]
    return np.stack([tl, tr, br, bl], axis=0)


def corners_from_contour(cnt: np.ndarray) -> np.ndarray:
    """컨투어에서 4코너 추출. 실패 시 minAreaRect 박스포인트로 대체."""
    peri = cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
    if len(approx) == 4:
        pts = approx.reshape(4, 2)
        return order_points_clockwise(pts)

    # fallback
    rect = cv2.minAreaRect(cnt)
    box = cv2.boxPoints(rect)  # 4x2
    return order_points_clockwise(box)


def scale_from_hinv(Hinv: np.ndarray, center_cm=(BOARD_W_CM/2, BOARD_H_CM/2)):
    """cm->pixel 변환(Hinv)으로 중심점에서 px/cm를 근사 계산"""
    cx, cy = center_cm
    p0 = np.array([[[cx, cy]]], dtype=np.float32)
    px = np.array([[[cx + 1.0, cy]]], dtype=np.float32)  # +1 cm in x
    py = np.array([[[cx, cy + 1.0]]], dtype=np.float32)  # +1 cm in y

    ip0 = cv2.perspectiveTransform(p0, Hinv)[0, 0]
    ipx = cv2.perspectiveTransform(px, Hinv)[0, 0]
    ipy = cv2.perspectiveTransform(py, Hinv)[0, 0]

    px_per_cm_x = float(np.linalg.norm(ipx - ip0))
    px_per_cm_y = float(np.linalg.norm(ipy - ip0))
    return px_per_cm_x, px_per_cm_y


def detect_board_in_roi(roi_bgr):
    """
    ROI 이미지에서 '검은 보드' 후보를 찾고 best 후보 반환

    반환:
      best_dict: {
        long_px, short_px, px_per_cm_x, px_per_cm_y,
        corners (TL,TR,BR,BL in ROI coords),
        H_pix2cm (pixel->cm), H_cm2pix
      }
      mask

    검출은 HSV(어두움+저채도) 기반.
    """
    h, w = roi_bgr.shape[:2]
    roi_area = float(h * w)

    hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
    Hc, S, V = cv2.split(hsv)

    # 블랙보드 후보 마스크: 어둡고(V 낮음) + 채도 낮음(S 낮음)
    mask_dark = (V < 85).astype(np.uint8) * 255
    mask_low_sat = (S < 90).astype(np.uint8) * 255
    mask = cv2.bitwise_and(mask_dark, mask_low_sat)

    kernel = np.ones((9, 9), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best = None
    best_score = -1e18

    # 면적 하한: ROI의 0.5% 미만은 버림
    min_area = 0.005 * roi_area

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue

        rect = cv2.minAreaRect(cnt)
        (rw, rh) = rect[1]
        if rw <= 1 or rh <= 1:
            continue

        long_px = float(max(rw, rh))
        short_px = float(min(rw, rh))
        ratio = long_px / short_px

        ratio_err = abs(ratio - TARGET_RATIO)
        rect_area = float(rw * rh)
        fill_ratio = area / rect_area if rect_area > 0 else 0.0
        if fill_ratio < 0.55:
            continue

        if not (1.10 <= ratio <= 1.75):
            continue

        score = (-ratio_err * 3.0) + (fill_ratio * 1.5) + (np.log(area + 1.0) * 0.25)

        if score > best_score:
            best_score = score

            # 코너 4점 추출 (ROI 좌표)
            corners = corners_from_contour(cnt)  # TL,TR,BR,BL

            # homography: pixel(ROI) -> cm
            dst_cm = np.array([
                [0.0, 0.0],
                [BOARD_W_CM, 0.0],
                [BOARD_W_CM, BOARD_H_CM],
                [0.0, BOARD_H_CM]
            ], dtype=np.float32)

            H_pix2cm = cv2.getPerspectiveTransform(corners.astype(np.float32), dst_cm)
            H_cm2pix = np.linalg.inv(H_pix2cm)

            # 기본(minAreaRect) 기반 px/cm (fallback)
            px_per_cm_x_rect = long_px / BOARD_W_CM
            px_per_cm_y_rect = short_px / BOARD_H_CM

            best = {
                "long_px": long_px,
                "short_px": short_px,
                "px_per_cm_x_rect": px_per_cm_x_rect,
                "px_per_cm_y_rect": px_per_cm_y_rect,
                "corners": corners,
                "H_pix2cm": H_pix2cm,
                "H_cm2pix": H_cm2pix,
            }

    if best is None:
        return None, mask

    # homography 기반 중심점 px/cm 계산
    try:
        px_per_cm_x_h, px_per_cm_y_h = scale_from_hinv(best["H_cm2pix"])
        best["px_per_cm_x_h"] = px_per_cm_x_h
        best["px_per_cm_y_h"] = px_per_cm_y_h
    except Exception:
        best["px_per_cm_x_h"] = None
        best["px_per_cm_y_h"] = None

    return best, mask


# =========================
# 4. 단일 이미지 처리
# =========================

def process_image(img_path):
    global _debug_saved

    try:
        img = cv2.imread(img_path)
        if img is None:
            return None

        cam_id = get_cam_id(os.path.basename(img_path))
        roi_img, (x_off, y_off) = crop_roi(img, cam_id)

        best, mask = detect_board_in_roi(roi_img)
        if best is None:
            # 디버그 저장(필요할 때만)
            if DEBUG_SAVE_FAILED and _debug_saved < MAX_DEBUG_SAVE:
                _ensure_dir(DEBUG_DIR)
                base = os.path.splitext(os.path.basename(img_path))[0]
                cv2.imwrite(os.path.join(DEBUG_DIR, f"{base}_roi.jpg"), roi_img)
                cv2.imwrite(os.path.join(DEBUG_DIR, f"{base}_mask.png"), mask)
                _debug_saved += 1
            return None

        # cam1은 homography 기반 px/cm 우선 사용
        if cam_id == 1 and best.get("px_per_cm_x_h") is not None:
            px_per_cm_x = best["px_per_cm_x_h"]
            px_per_cm_y = best["px_per_cm_y_h"]
        else:
            px_per_cm_x = best["px_per_cm_x_rect"]
            px_per_cm_y = best["px_per_cm_y_rect"]

        long_px = best["long_px"]
        short_px = best["short_px"]

        return {
            "image_name": os.path.basename(img_path),
            "cam": cam_id,
            "width_px": long_px,
            "height_px": short_px,
            "px_per_cm_x": px_per_cm_x,
            "px_per_cm_y": px_per_cm_y,
        }

    except Exception:
        return None


# =========================
# 5. 전체 실행
# =========================

def run_pixel_scale_extraction(root_dir, output_csv):
    # 이미지 파일 전체 수집
    img_files = []
    for root, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith((".jpg", ".png")):
                img_files.append(os.path.join(root, f))

    results = []

    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = [executor.submit(process_image, p) for p in img_files]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing"):
            res = future.result()
            if res is not None:
                results.append(res)

    import pandas as pd
    df = pd.DataFrame(results)

    # -----------------------------
    # cam1 → bed_date 단위 median pixel_scale 적용
    # cam0 → 프레임 단위 유지
    # -----------------------------

    # bed_date 추출 (예: bed00_20251213)
    def extract_bed_date(name):
        base = os.path.splitext(name)[0]
        return "_".join(base.split("_")[:2])

    df["bed_date"] = df["image_name"].apply(extract_bed_date)

    # cam1 median 스케일 계산
    cam1_median = (
        df[df["cam"] == 1]
        .groupby("bed_date")[["px_per_cm_x", "px_per_cm_y"]]
        .median()
        .reset_index()
        .rename(columns={
            "px_per_cm_x": "px_per_cm_x_median",
            "px_per_cm_y": "px_per_cm_y_median",
        })
    )

    # 병합
    df = df.merge(cam1_median, on="bed_date", how="left")

    # cam1은 median 사용, cam0은 원래 값 유지
    df.loc[df["cam"] == 1, "px_per_cm_x"] = df.loc[df["cam"] == 1, "px_per_cm_x_median"]
    df.loc[df["cam"] == 1, "px_per_cm_y"] = df.loc[df["cam"] == 1, "px_per_cm_y_median"]

    # 보조 컬럼 제거
    df.drop(columns=["px_per_cm_x_median", "px_per_cm_y_median"], inplace=True)

    # CSV 저장
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print(f"Done: {len(df)} / {len(img_files)} images succeeded")


In [3]:

# =========================
# 6. 실행 예시
# =========================
if __name__ == "__main__":
    TEST_ROOT = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251213-251226/"   # ← 네 경로로 수정
    OUTPUT_CSV = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251213-251226/260102_pixel_scale_final.csv"

    run_pixel_scale_extraction(TEST_ROOT, OUTPUT_CSV)



Processing: 100%|██████████| 13917/13917 [38:13<00:00,  6.07it/s]


Done: 8279 / 13917 images succeeded


In [17]:
df2=pd.read_csv("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251213-251226/251213_pixel_scale_Cam1median.csv")
def cv(x):
    return (x.std() / x.mean()) * 100

# mean, std, cv 계산
df2.groupby("cam")[["px_per_cm_x","px_per_cm_y"]].agg(["mean","std", cv])

px_per_cm_x                      px_per_cm_y                     
           mean       std         cv        mean       std         cv
cam                                                                  
0      9.003064  0.769572   8.547894    9.845512  0.460289   4.675120
1      9.908456  2.077059  20.962494    9.985396  1.698141  17.006245